# AuthentiScan — A4: cross-generator sweep + Grad-CAM galleries

**First run of this notebook?** Kaggle Secrets attach per notebook: open Add-ons →
Secrets and toggle `GITHUB_PAT` ON for this notebook first, or cell 1 fails with
'No user secrets exist'.

Evaluation only — no training. Inputs: CIFAKE, `yangsangtai/tiny-genimage`, and the
outputs of A3 sessions 1–8 (every canonical checkpoint). Cross-generator: 10 canonical
checkpoints × 4 generators × 2 conditions = 80 `crossgen.csv` rows. Grad-CAM: five
better checkpoints, shared correct set. Estimated wall-clock 1–2 h (an estimate — image
decoding dominates, not the GPU); any GPU is fine here, results do not depend on it.

**After it finishes:** download `results_a4.zip` from the Output tab and send it
back — `crossgen.csv` merges with `code/merge_runs.py --kind crossgen`, never retyped.


In [ ]:
# 1. Code: clone the private repo (Kaggle Secret GITHUB_PAT) or pull if already there
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT = UserSecretsClient().get_secret("GITHUB_PAT")
REPO_DIR = "/kaggle/working/sm7"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
                    f"https://{PAT}@github.com/rohityaduvxnshi/sm7.git", REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(f"{REPO_DIR}/sem8_major")
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)


In [ ]:
# 2. Extras (never upgrade Kaggle's torch/torchvision), then find the three mounts.
!pip install -q timm grad-cam

import os, sys, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
# Evaluation only: metrics do not depend on the GPU model, so no T4 assertion here.

DATA = GEN = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if DATA is None and os.path.isdir(os.path.join(root, "train", "REAL")):
        DATA = root
    if GEN is None and any(d.lower() == "imagenet_midjourney" or
                           d.lower().startswith("imagenet_ai_") for d in dirs):
        GEN = root
    if root.count(os.sep) > 5:   # don't descend into image folders
        dirs.clear()
assert DATA, "CIFAKE not found under /kaggle/input - is the dataset attached?"
assert GEN, "tiny-genimage not found under /kaggle/input - is the dataset attached?"
print("CIFAKE:      ", DATA)
print("tiny-genimage:", GEN)

sys.path.insert(0, "code")
from eval import find_checkpoints, read_manifest
CK = find_checkpoints("/kaggle/input")
IDS = read_manifest("results/canonical_runs.txt")
for rid in IDS:
    print(("OK      " if rid in CK else "MISSING ") + rid, CK.get(rid, ""))
missing = [r for r in IDS if r not in CK]
assert not missing, f"attach the A3 session outputs that hold: {missing}"


In [ ]:
# 3. Cross-generator sweep: verifies tiny-genimage contents first, then 80 cells.
# One eval.py subprocess per cell; a failure does not stop the rest.
!python code/run_crossgen.py --checkpoint-root /kaggle/input \
    --genimage-root "{GEN}" --results-dir /kaggle/working/results


In [ ]:
# 4. Grad-CAM galleries: five better checkpoints, shared correct set (plan 7a A1.5/A4).
!python code/make_galleries.py --checkpoint-root /kaggle/input \
    --data-root "{DATA}" --results-dir /kaggle/working/results \
    --out-root /kaggle/working/figures/gradcam


In [ ]:
# 5. What this session produced
import pandas as pd
cg = pd.read_csv("/kaggle/working/results/crossgen.csv")
print(len(cg), "crossgen rows (expected 80)")
display(cg.pivot_table(index="run_id", columns=["condition", "generator"],
                       values="acc").round(4))
display(pd.read_csv("/kaggle/working/results/genimage_verification.csv"))
print("galleries:", sorted(os.listdir("/kaggle/working/figures/gradcam")))


In [ ]:
# 6. Package for download: crossgen rows + per-cell artefacts + galleries (no checkpoints)
!cd /kaggle/working && zip -qr results_a4.zip results figures && ls -lh results_a4.zip
